## Investigate the effect of fraction_kinetic_target on the homeostatic term. Try to see how to meet the kinetic target at the cost of the homeostatic term

In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
# import sim
time = '1000'
date = '2026-05-06'
experiment = 'bob_index5328'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [3]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

print(len(homeostatic_dm_targets))
len(homeostatic_metabolite_counts)

172


172

In [4]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]

def test_NetworkFlowModel(weights, solver_choice = cp.GLOP,
                          homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                          homeostatic_dm_targets = homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          fraction_kinetic_target=1,
                          maintenance=maintenance,
                          ):
    model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts, # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())), # conc
            maintenance_target=maintenance, # conc range
            kinetic_targets=np.array(list(dict(kinetic).values())), # conc range
            binary_kinetic_idx=binary_kinetics_idx,
            # force_flow_idx=force_reaction_idx,
            objective_weights=weights, #same
            upper_flux_bound= 100, # increase to 10^9 because notebook runs FlowResult using Counts if directly imported from listeners, WC runs using conc.
            target_minimal_flux=counts_to_molar[-1],
            fraction_kinetic_target = fraction_kinetic_target,
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.homeostatic_term, solution.kinetics_term, solution.velocities, solution.dm_dt

In [42]:
objective_weights = {
    "secretion": 2.01E-07,
    "efficiency": 7.74E-06,
    "kinetics": 3.22447E-05,
    "diversity": 0.000954656,
    "homeostatic": 0.58035203
}

kinetic_fraction_range = np.linspace(0, 1, 21)
output = dict()
df_flux  = pd.DataFrame(index=reaction_names)
for kinetic_fraction in kinetic_fraction_range:

    try :
        obj_val, home_term, kin_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                      fraction_kinetic_target=kinetic_fraction)

        output[kinetic_fraction] = dict()
        output[kinetic_fraction]['Weighted Homeostatic Term'] = home_term
        output[kinetic_fraction]['Unweighted Homeostatic Term'] = home_term/objective_weights['homeostatic']
        # output[kinetic_fraction]['Weighted Kinetic Term'] = kin_term

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        df_flux[f'kinetic_fraction={kinetic_fraction}'] = flux

    except:
        continue

In [43]:
# display output as fancy table
output_df = pd.DataFrame(output).T
output_df = output_df.reset_index().rename(columns={'index': 'Kinetic Fraction'})
output_df = output_df.melt(id_vars='Kinetic Fraction', var_name='Term Type', value_name='Term Value')
fig = px.line(output_df, x='Kinetic Fraction', y='Term Value', color
='Term Type', markers=True, title='Effect of Fraction Kinetic Target on Homeostatic and Kinetic Terms')
fig.show()

In [44]:
# locally perturb the objective weights to see if we can meet the kinetic target at the cost of the homeostatic term
import numpy as np

# kinetic reaction indices in stoichiometry
kin_names = set(metabolism.kinetic_constraint_reactions)
hom_names = set(metabolism.homeostatic_metabolites)

kin_idx = [i for i, r in enumerate(metabolism.reaction_names) if r in kin_names]
hom_idx = [i for i, m in enumerate(metabolism.metabolite_names) if m in hom_names]

# sub-stoichiometry: rows = homeostatic metabolites, cols = kinetic reactions
sub_S = stoichiometry[np.ix_(hom_idx, kin_idx)]
connected_hom = np.any(sub_S != 0, axis=1)  # which homeostatic mets are touched by kinetic rxns
connected_kin = np.any(sub_S != 0, axis=0)  # which kinetic rxns touch homeostatic mets

print(f"{connected_hom.sum()} / {len(hom_idx)} homeostatic metabolites stoichiometrically connected to kinetic reactions")
print(f"{connected_kin.sum()} / {len(kin_idx)} kinetic reactions touch at least one homeostatic metabolite")

125 / 172 homeostatic metabolites stoichiometrically connected to kinetic reactions
393 / 416 kinetic reactions touch at least one homeostatic metabolite


In [45]:
import warnings
import polars as pl
import numpy as np
import cvxpy as cp
from tqdm import tqdm
from joblib import Parallel, delayed
import os, sys

sys.path.insert(0, os.path.expanduser("~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes"))
os.chdir(os.path.expanduser("~/dev/vEcoli"))

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FREE_RXNS

# ── USER CONFIG ────────────────────────────────────────────────────────────────
CSV_PATH = ("notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/"
            "modified_from_csv_1/filtered_results.csv")
FRACTIONS = [1.0, 0.5, 0.1]   # add more if desired, e.g. [1.0, 0.75, 0.5, 0.25]
OUT_DIR   = "notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/knockdown"
N_JOBS    = 10
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

weight_df = pl.read_csv(CSV_PATH)
required = {"lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"}
assert required.issubset(set(weight_df.columns)), f"Missing columns: {required - set(weight_df.columns)}"
if "lambda_hom" not in weight_df.columns:
    weight_df = weight_df.with_columns(pl.lit(1.0).alias("lambda_hom"))
    print("Note: lambda_hom not found — defaulting to 1.0")

weight_samples = weight_df.select(
    ["lambda_hom", "lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
).to_numpy()
print(f"Loaded {len(weight_samples)} weight combinations")


def _solve(i, fraction):
    lam_hom, lam_sec, lam_eff, lam_kin, lam_div = weight_samples[i]
    try:
        model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS,
        )
        model.set_up_exchanges(
            exchanges=metabolism.exchange_molecules,
            uptakes=metabolism.allowed_exchange_uptake,
        )
        sol = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts,
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())),
            maintenance_target=maintenance,
            kinetic_targets=np.array(list(dict(kinetic).values())),
            objective_weights={
                "homeostatic": lam_hom, "secretion": lam_sec,
                "efficiency": lam_eff, "kinetics": lam_kin, "diversity": lam_div,
            },
            upper_flux_bound=100,
            target_minimal_flux=counts_to_molar[-1],
            fraction_kinetic_target=fraction,
            solver=cp.GLOP,
        )
        return {
            "idx": i, "fraction": fraction,
            "lambda_hom": lam_hom, "lambda_sec": lam_sec,
            "lambda_eff": lam_eff, "lambda_kin": lam_kin, "lambda_div": lam_div,
            "obj_homeo": sol.homeostatic_term,
            "obj_kin":   sol.kinetics_term,
            "obj_eff":   sol.efficiency_term,
            "obj_sec":   sol.secretion_term,
            "obj_div":   sol.diversity_term,
            "obj_total": sol.objective,
        }
    except Exception:
        return None


all_rows = []
for frac in FRACTIONS:
    print(f"\nSolving at fraction_kinetic_target = {frac} ...")
    if N_JOBS == 1:
        rows = [_solve(i, frac) for i in tqdm(range(len(weight_samples)))]
    else:
        rows = Parallel(n_jobs=N_JOBS)(
            delayed(_solve)(i, frac) for i in tqdm(range(len(weight_samples)))
        )
    valid = [r for r in rows if r is not None]
    n_failed = len(weight_samples) - len(valid)
    if n_failed:
        warnings.warn(f"  {n_failed} solves failed at fraction={frac}")
    print(f"  {len(valid)} successful")
    all_rows.extend(valid)

results_df = pl.DataFrame(all_rows)
results_df.write_csv(f"{OUT_DIR}/knockdown_results.csv")
print(f"\nSaved: {OUT_DIR}/knockdown_results.csv")

Loaded 544 weight combinations

Solving at fraction_kinetic_target = 1.0 ...


  6%|▌         | 30/544 [00:03<00:54,  9.39it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.

 17%|█▋        | 90/544 [00:13<01:06,  6.79it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 28%|██▊       | 150/544 [00:22<00:58,  6.69it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 31%|███▏      | 170/544 [00:25<00:57,  6.50it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce


  544 successful

Solving at fraction_kinetic_target = 0.5 ...


 17%|█▋        | 90/544 [00:12<01:06,  6.82it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 24%|██▍       | 130/544 [00:18<00:59,  6.98it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 51%|█████▏    | 280/544 [00:40<00:40,  6.59it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 53%|█████▎    | 290/544 [00:42<00:37,  6.85it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, 

  544 successful

Solving at fraction_kinetic_target = 0.1 ...


 11%|█         | 60/544 [00:07<01:07,  7.13it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 17%|█▋        | 90/544 [00:12<01:07,  6.74it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 18%|█▊        | 100/544 [00:13<01:04,  6.93it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
 20%|██        | 110/544 [00:15<01:06,  6.51it/s]/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, *

  544 successful

Saved: notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/knockdown/knockdown_results.csv


In [55]:
results_pd = results_df.to_pandas()

frac_hi, frac_lo = FRACTIONS[0], FRACTIONS[-1]   # e.g. 1.0 and 0.1

pivot = results_pd.pivot_table(
    index=["idx", "lambda_hom", "lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"],
    columns="fraction",
    values=["obj_homeo", "obj_kin", "obj_eff", "obj_sec", "obj_total"],
).reset_index()
pivot.columns = ["_".join(str(c) for c in col).rstrip("_") for col in pivot.columns]

pivot["delta_homeo"] = pivot[f"obj_homeo_{frac_lo}"] - pivot[f"obj_homeo_{frac_hi}"]
pivot["delta_kin"]   = pivot[f"obj_kin_{frac_lo}"]   - pivot[f"obj_kin_{frac_hi}"]
pivot["delta_eff"]   = pivot[f"obj_eff_{frac_lo}"]   - pivot[f"obj_eff_{frac_hi}"]

pivot["log_lambda_hom"] = np.log10(pivot["lambda_hom"])
pivot["log_lambda_kin"] = np.log10(pivot["lambda_kin"])
pivot["log_lambda_eff"] = np.log10(pivot["lambda_eff"])
pivot["log_kin_hom_ratio"] = np.log10(pivot["lambda_kin"] / pivot["lambda_hom"])

pivot.to_csv(f"{OUT_DIR}/knockdown_delta.csv", index=False)
print(f"Shape: {pivot.shape}")
print(pivot[["lambda_hom", "lambda_kin", "delta_homeo", "delta_kin"]].describe())

Shape: (544, 28)
       lambda_hom    lambda_kin   delta_homeo   delta_kin
count  544.000000  5.440000e+02  5.440000e+02  544.000000
mean     0.409095  8.874389e-06  1.867137e-13   -9.845641
std      0.278784  9.340397e-06  2.619065e-12    2.067418
min      0.013260  6.330000e-07 -1.056260e-11  -13.176486
25%      0.156111  1.977500e-06 -1.368269e-12  -11.337732
50%      0.353633  5.285000e-06  1.965054e-13  -11.301974
75%      0.640759  1.243877e-05  1.900871e-12   -7.784759
max      0.993835  4.346430e-05  7.392758e-12   -7.710682


In [56]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

HOVER = {
    "lambda_hom": ":.2e", "lambda_kin": ":.2e", "lambda_eff": ":.2e",
    "delta_homeo": ":.3e", "delta_kin": ":.3e",
}

# ── Plot 1: Δhomeo vs Δkin — the key diagnostic scatter ──────────────────────
# Quadrant of interest: bottom-left = Δkin < 0 (kinetics easier) AND Δhomeo > 0
# (homeostasis more costly — real compensation signal)
fig1 = px.scatter(
    pivot, x="delta_kin", y="delta_homeo",
    color="log_kin_hom_ratio",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    hover_data=HOVER,
    title=(f"Δobj_homeo vs Δobj_kin  (fraction: {frac_hi} → {frac_lo})<br>"
           "<sup>Color = log₁₀(λ_kin / λ_hom) — "
           "target region: Δkin < 0 AND Δhomeo large</sup>"),
    labels={
        "delta_kin":   f"Δobj_kin ({frac_hi}→{frac_lo})",
        "delta_homeo": f"Δobj_homeo ({frac_hi}→{frac_lo})",
        "log_kin_hom_ratio": "log₁₀(λ_kin/λ_hom)",
    },
    template="plotly_white",
)
fig1.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig1.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.4)
# fig1.write_html(f"{OUT_DIR}/delta_scatter.html")
fig1.show()

# ── Plot 2: log(λ_kin) × log(λ_hom) heatmap colored by Δhomeo ───────────────
fig2 = px.scatter(
    pivot, x="log_lambda_kin", y="log_lambda_hom",
    color="delta_homeo",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    hover_data=HOVER,
    title=(f"λ_kin × λ_hom space colored by Δobj_homeo<br>"
           "<sup>Reveals the compensation regime — "
           "warm = knockdown disrupts homeostasis</sup>"),
    labels={
        "log_lambda_kin": "log₁₀(λ_kin)",
        "log_lambda_hom": "log₁₀(λ_hom)",
        "delta_homeo": "Δobj_homeo",
    },
    template="plotly_white",
)
# fig2.write_html(f"{OUT_DIR}/heatmap_kin_hom.html")
fig2.show()

# ── Plot 3: Same heatmap but for Δkin, so you can verify kinetics DID respond ─
fig3 = px.scatter(
    pivot, x="log_lambda_kin", y="log_lambda_hom",
    color="delta_kin",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    hover_data=HOVER,
    title="λ_kin × λ_hom colored by Δobj_kin  (sanity check)",
    labels={
        "log_lambda_kin": "log₁₀(λ_kin)", "log_lambda_hom": "log₁₀(λ_hom)",
        "delta_kin": "Δobj_kin",
    },
    template="plotly_white",
)
# fig3.write_html(f"{OUT_DIR}/heatmap_kin_hom_deltakin.html")
fig3.show()

# ── Table: top candidates (Δkin < 0, largest Δhomeo) ─────────────────────────
candidates = (
    pivot[pivot["delta_kin"] < 0]
    .sort_values("delta_homeo", ascending=False)
)
print(f"\n{len(candidates)} weight combos where knockdown made kinetics easier (Δkin < 0)")
print("\nTop 15 by Δobj_homeo (strongest compensation signal):")
cols = ["lambda_hom", "lambda_kin", "lambda_eff", "lambda_sec",
        "delta_homeo", "delta_kin",
        f"obj_homeo_{frac_hi}", f"obj_homeo_{frac_lo}"]
print(candidates[cols].head(15).to_string(index=False, float_format="%.3e"))
candidates[cols].head(50).to_csv(f"{OUT_DIR}/top_candidates.csv", index=False)
print(f"Saved top 50 to {OUT_DIR}/top_candidates.csv")


544 weight combos where knockdown made kinetics easier (Δkin < 0)

Top 15 by Δobj_homeo (strongest compensation signal):
 lambda_hom  lambda_kin  lambda_eff  lambda_sec  delta_homeo  delta_kin  obj_homeo_1.0  obj_homeo_0.1
  7.048e-01   8.960e-06   1.750e-06   1.110e-07    7.393e-12 -7.785e+00      3.360e-12      1.075e-11
  3.248e-01   1.490e-05   3.040e-06   1.400e-06    7.176e-12 -1.132e+01      5.120e-12      1.230e-11
  6.790e-01   4.010e-06   7.360e-07   2.360e-07    7.173e-12 -7.784e+00      1.032e-12      8.205e-12
  1.080e-01   5.960e-06   1.130e-06   8.350e-07    6.845e-12 -1.132e+01      1.989e-12      8.834e-12
  9.047e-01   1.252e-05   2.520e-06   6.910e-07    6.272e-12 -7.787e+00      1.092e-12      7.364e-12
  1.277e-01   1.800e-06   3.100e-07   6.800e-07    6.186e-12 -1.311e+01      4.197e-12      1.038e-11
  3.905e-01   1.100e-06   1.820e-07   4.010e-07    6.152e-12 -1.219e+01      5.284e-12      1.144e-11
  1.057e-01   6.330e-07   1.000e-07   1.150e-07    5.611e-12 -

In [57]:
weight_df

Index,lambda_hom,lambda_sec,lambda_eff,lambda_kin,lambda_div,obj_total,obj_homeo,obj_kin,obj_eff,obj_sec,obj_div,obj_hom_w,obj_kin_w,obj_eff_w,obj_sec_w,obj_div_w,toya_pearson_r_squared,toya_r_squared
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
3,0.376525,4.0600e-7,1.3300e-7,8.2200e-7,0.000021,0.000041,2.4700e-8,22.902749,148.099543,6.361307,0.003617,9.3100e-9,0.000019,0.00002,0.000003,7.5900e-8,0.734228,0.659225
4,0.123624,1.2900e-7,5.4300e-7,0.000003,0.000011,0.0001506,2.4700e-8,22.787188,148.646668,6.471077,0.0037038,3.0600e-9,0.000069,0.000081,8.3500e-7,3.9500e-8,0.734851,0.659726
6,0.84502,2.9600e-7,0.000002,0.000009,0.000923,0.000472,2.4700e-8,22.836682,148.385338,6.449439,0.003617,2.0900e-8,0.000205,0.000261,0.000002,0.000003,0.734361,0.659624
14,0.293958,3.0600e-7,0.000003,0.000016,0.000024,0.000846,2.4700e-8,22.888433,148.130724,6.379163,0.0041288,7.2700e-9,0.0003628,0.000481,0.000002,1.0100e-7,0.733598,0.658339
23,0.816759,8.6200e-7,5.2900e-7,0.000003,0.000241,0.000152,2.4700e-8,22.891526,148.123872,6.377187,0.003617,2.0200e-8,0.000068,0.000078,0.0000055,8.7100e-7,0.733568,0.658355
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
9908,0.230717,4.2300e-7,5.2300e-7,0.000003,0.000044,0.000147,2.4700e-8,22.888268,148.139586,6.379279,0.003618,5.7100e-9,0.000067,0.000077,0.0000027,1.6000e-7,0.733972,0.659192
9940,0.131328,0.000001,5.6100e-7,0.000003,0.000027,0.000164,2.4700e-8,22.901258,148.097758,6.364692,0.003631,3.2500e-9,0.000071,0.000083,0.0000091,9.6900e-8,0.734062,0.65908
9957,0.171301,1.6400e-7,0.0000013,0.000007,0.000063,0.00035,2.4700e-8,22.836668,148.384918,6.449367,0.003624,4.2300e-9,0.000156,0.0001929,0.000001,2.2800e-7,0.734359,0.659623


## Per Riley's Suggestion: Try doing the kinetic target search on just homeostatic and kinetic weights only.

In [57]:
import warnings
import polars as pl
import numpy as np
import cvxpy as cp
from tqdm import tqdm
from joblib import Parallel, delayed
import os, sys

sys.path.insert(0, os.path.expanduser("~/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes"))
os.chdir(os.path.expanduser("~/dev/vEcoli"))

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FREE_RXNS

# ── USER CONFIG ────────────────────────────────────────────────────────────────
FRACTIONS = [1.0, 0.5, 0.1]   # add more if desired, e.g. [1.0, 0.75, 0.5, 0.25]
OUT_DIR   = "notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_10000samples/knockdown"
N_JOBS    = 10
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

# ── Sampling lambda_hom and lambda_kin  ───────────────────────────────────────────────
weight_df = pl.read_csv(CSV_PATH)
required = {"lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"}
assert required.issubset(set(weight_df.columns)), f"Missing columns: {required - set(weight_df.columns)}"
if "lambda_hom" not in weight_df.columns:
    weight_df = weight_df.with_columns(pl.lit(1.0).alias("lambda_hom"))
    print("Note: lambda_hom not found — defaulting to 1.0")

weight_samples = weight_df.select(
    ["lambda_hom", "lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
).to_numpy()
print(f"Loaded {len(weight_samples)} weight combinations")


def _solve(i, fraction):
    lam_hom, lam_sec, lam_eff, lam_kin, lam_div = weight_samples[i]
    try:
        model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS,
        )
        model.set_up_exchanges(
            exchanges=metabolism.exchange_molecules,
            uptakes=metabolism.allowed_exchange_uptake,
        )
        sol = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts,
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())),
            maintenance_target=maintenance,
            kinetic_targets=np.array(list(dict(kinetic).values())),
            objective_weights={
                "homeostatic": lam_hom, "secretion": lam_sec,
                "efficiency": lam_eff, "kinetics": lam_kin, "diversity": lam_div,
            },
            upper_flux_bound=100,
            target_minimal_flux=counts_to_molar[-1],
            fraction_kinetic_target=fraction,
            solver=cp.GLOP,
        )
        return {
            "idx": i, "fraction": fraction,
            "lambda_hom": lam_hom, "lambda_sec": lam_sec,
            "lambda_eff": lam_eff, "lambda_kin": lam_kin, "lambda_div": lam_div,
            "obj_homeo": sol.homeostatic_term,
            "obj_kin":   sol.kinetics_term,
            "obj_eff":   sol.efficiency_term,
            "obj_sec":   sol.secretion_term,
            "obj_div":   sol.diversity_term,
            "obj_total": sol.objective,
        }
    except Exception:
        return None


all_rows = []
for frac in FRACTIONS:
    print(f"\nSolving at fraction_kinetic_target = {frac} ...")
    if N_JOBS == 1:
        rows = [_solve(i, frac) for i in tqdm(range(len(weight_samples)))]
    else:
        rows = Parallel(n_jobs=N_JOBS)(
            delayed(_solve)(i, frac) for i in tqdm(range(len(weight_samples)))
        )
    valid = [r for r in rows if r is not None]
    n_failed = len(weight_samples) - len(valid)
    if n_failed:
        warnings.warn(f"  {n_failed} solves failed at fraction={frac}")
    print(f"  {len(valid)} successful")
    all_rows.extend(valid)

results_df = pl.DataFrame(all_rows)
results_df.write_csv(f"{OUT_DIR}/knockdown_results.csv")
print(f"\nSaved: {OUT_DIR}/knockdown_results.csv")